### Gold PipelineBuilds the aggregated gold tables that answer the dashboard's fraud-analysis questions, sourced entirely from the three silver tables. Run this as the `Gold` task, after `Silver`.

In [0]:
%sqlCREATE SCHEMA IF NOT EXISTS catalog.gold;

In [0]:
from pyspark.sql import functions as Ffrom pyspark.sql.window import Window

In [0]:
transactions_df = spark.read.table("catalog.silver.transactions")mcc_codes_df = spark.read.table("catalog.silver.mcc_codes")fraud_labels_df = spark.read.table("catalog.silver.fraud_labels")fraud_df = transactions_df.filter(F.col("is_fraud") == True)

**Q1: Which day(s) of the week see the highest number of fraudulent transactions?**

In [0]:
gold_fraud_by_day_of_week_df = (fraud_df    .groupBy("day_of_week", "day_of_week_num")    .agg(F.count("*").alias("fraud_txn_count"))    .orderBy(F.desc("fraud_txn_count")))gold_fraud_by_day_of_week_df.write.mode("overwrite").saveAsTable("catalog.gold.fraud_by_day_of_week")display(gold_fraud_by_day_of_week_df)

**Q2: What is the trend of the fraud rate over the past month?**

In [0]:
max_date = transactions_df.agg(F.max("txn_date")).first()[0]gold_fraud_rate_trend_df = (transactions_df    .filter(F.col("txn_date") >= F.date_sub(F.lit(max_date), 30))    .groupBy("txn_date")    .agg(        F.count("*").alias("total_txn_count"),        F.sum(F.col("is_fraud").cast("int")).alias("fraud_txn_count")    )    .withColumn("fraud_rate", F.col("fraud_txn_count") / F.col("total_txn_count"))    .orderBy("txn_date"))gold_fraud_rate_trend_df.write.mode("overwrite").saveAsTable("catalog.gold.fraud_rate_trend")display(gold_fraud_rate_trend_df)

**Q3: Which users have the largest number of flagged (`is_fraud = true`) transactions?**

In [0]:
gold_top_fraud_users_df = (fraud_df    .groupBy("client_id")    .agg(F.count("*").alias("fraud_txn_count"))    .orderBy(F.desc("fraud_txn_count")))gold_top_fraud_users_df.write.mode("overwrite").saveAsTable("catalog.gold.top_fraud_users")display(gold_top_fraud_users_df)

**Q4: Are there any users showing a sharp rise in transaction amount compared to their weekly average?**"Sharp rise" is defined here as a transaction whose amount is more than 2x the user's trailing weekly average amount (excluding the current transaction). Adjust the multiplier as needed.

In [0]:
user_week_window = Window.partitionBy("client_id", "week_of_year", "year_month")txn_with_week_avg_df = (transactions_df    .withColumn(        "weekly_avg_amount",        (F.sum("amount").over(user_week_window) - F.col("amount"))        / F.greatest(F.count("amount").over(user_week_window) - F.lit(1), F.lit(1))    ))gold_sharp_rise_users_df = (txn_with_week_avg_df    .filter(F.col("weekly_avg_amount") > 0)    .withColumn("pct_above_weekly_avg", (F.col("amount") - F.col("weekly_avg_amount")) / F.col("weekly_avg_amount"))    .filter(F.col("amount") > 2 * F.col("weekly_avg_amount"))    .select("id", "client_id", "txn_date", "amount", "weekly_avg_amount", "pct_above_weekly_avg", "is_fraud")    .orderBy(F.desc("pct_above_weekly_avg")))gold_sharp_rise_users_df.write.mode("overwrite").saveAsTable("catalog.gold.sharp_rise_users")display(gold_sharp_rise_users_df)

**Q5: Which merchant categories (MCC) exhibit the highest fraud rate?**

In [0]:
gold_fraud_rate_by_mcc_df = (transactions_df    .groupBy("mcc", "mcc_description")    .agg(        F.count("*").alias("total_txn_count"),        F.sum(F.col("is_fraud").cast("int")).alias("fraud_txn_count")    )    .withColumn("fraud_rate", F.col("fraud_txn_count") / F.col("total_txn_count"))    .orderBy(F.desc("fraud_rate")))gold_fraud_rate_by_mcc_df.write.mode("overwrite").saveAsTable("catalog.gold.fraud_rate_by_mcc")display(gold_fraud_rate_by_mcc_df)

**Q6: Are there specific merchants with unusually high fraud volume?**

In [0]:
gold_fraud_by_merchant_df = (transactions_df    .groupBy("merchant_id", "merchant_city", "merchant_state")    .agg(        F.count("*").alias("total_txn_count"),        F.sum(F.col("is_fraud").cast("int")).alias("fraud_txn_count")    )    .withColumn("fraud_rate", F.col("fraud_txn_count") / F.col("total_txn_count"))    .orderBy(F.desc("fraud_txn_count")))gold_fraud_by_merchant_df.write.mode("overwrite").saveAsTable("catalog.gold.fraud_by_merchant")display(gold_fraud_by_merchant_df)

**Q7: How does fraud distribution vary by time of day (morning vs night)?**

In [0]:
gold_fraud_by_time_of_day_df = (transactions_df    .groupBy("time_of_day")    .agg(        F.count("*").alias("total_txn_count"),        F.sum(F.col("is_fraud").cast("int")).alias("fraud_txn_count")    )    .withColumn("fraud_rate", F.col("fraud_txn_count") / F.col("total_txn_count"))    .orderBy(F.desc("fraud_txn_count")))gold_fraud_by_time_of_day_df.write.mode("overwrite").saveAsTable("catalog.gold.fraud_by_time_of_day")display(gold_fraud_by_time_of_day_df)

**Q8: What's the average transaction amount for fraud vs non-fraud transactions?**

In [0]:
gold_avg_amount_fraud_vs_non_df = (transactions_df    .groupBy("is_fraud")    .agg(        F.avg("amount").alias("avg_amount"),        F.count("*").alias("txn_count")    ))gold_avg_amount_fraud_vs_non_df.write.mode("overwrite").saveAsTable("catalog.gold.avg_amount_fraud_vs_non")display(gold_avg_amount_fraud_vs_non_df)

**Q9: Which merchant category has the highest total fraud amount?**

In [0]:
gold_fraud_amount_by_mcc_df = (fraud_df    .groupBy("mcc", "mcc_description")    .agg(F.sum("amount").alias("total_fraud_amount"))    .orderBy(F.desc("total_fraud_amount")))gold_fraud_amount_by_mcc_df.write.mode("overwrite").saveAsTable("catalog.gold.fraud_amount_by_mcc")display(gold_fraud_amount_by_mcc_df)

**Q10: What are the total monetary losses due to fraud each day?**

In [0]:
gold_daily_fraud_losses_df = (fraud_df    .groupBy("txn_date")    .agg(F.sum("amount").alias("total_fraud_loss"))    .orderBy("txn_date"))gold_daily_fraud_losses_df.write.mode("overwrite").saveAsTable("catalog.gold.daily_fraud_losses")display(gold_daily_fraud_losses_df)

**Q11: How many unique users commit fraudulent transactions per week?**

In [0]:
gold_weekly_unique_fraud_users_df = (fraud_df    .groupBy("year_month", "week_of_year")    .agg(F.countDistinct("client_id").alias("unique_fraud_user_count"))    .orderBy("year_month", "week_of_year"))gold_weekly_unique_fraud_users_df.write.mode("overwrite").saveAsTable("catalog.gold.weekly_unique_fraud_users")display(gold_weekly_unique_fraud_users_df)

**Q12: Do fraud patterns show seasonal or monthly spikes?**

In [0]:
gold_monthly_fraud_pattern_df = (transactions_df    .groupBy("year_month")    .agg(        F.count("*").alias("total_txn_count"),        F.sum(F.col("is_fraud").cast("int")).alias("fraud_txn_count")    )    .withColumn("fraud_rate", F.col("fraud_txn_count") / F.col("total_txn_count"))    .orderBy("year_month"))gold_monthly_fraud_pattern_df.write.mode("overwrite").saveAsTable("catalog.gold.monthly_fraud_pattern")display(gold_monthly_fraud_pattern_df)

**Q13: How has user behavior changed before versus after a fraudulent event?**For each user's first fraud event, this compares their average transaction amount and transaction count in the 30 days before vs. the 30 days after.

In [0]:
first_fraud_event_df = (fraud_df    .groupBy("client_id")    .agg(F.min("txn_timestamp").alias("first_fraud_ts")))txn_with_fraud_event_df = transactions_df.join(first_fraud_event_df, on="client_id", how="inner")before_df = (txn_with_fraud_event_df    .filter(        (F.col("txn_timestamp") < F.col("first_fraud_ts")) &        (F.col("txn_timestamp") >= F.date_sub(F.col("first_fraud_ts"), 30))    )    .groupBy("client_id")    .agg(F.avg("amount").alias("avg_amount_before"), F.count("*").alias("txn_count_before")))after_df = (txn_with_fraud_event_df    .filter(        (F.col("txn_timestamp") > F.col("first_fraud_ts")) &        (F.col("txn_timestamp") <= F.date_add(F.col("first_fraud_ts"), 30))    )    .groupBy("client_id")    .agg(F.avg("amount").alias("avg_amount_after"), F.count("*").alias("txn_count_after")))gold_behavior_before_after_fraud_df = (before_df    .join(after_df, on="client_id", how="outer")    .withColumn("amount_change_pct",        (F.col("avg_amount_after") - F.col("avg_amount_before")) / F.col("avg_amount_before")))gold_behavior_before_after_fraud_df.write.mode("overwrite").saveAsTable("catalog.gold.behavior_before_after_fraud")display(gold_behavior_before_after_fraud_df)

**Q14: Are fraudulent transactions more common on high-value purchases compared to low-value purchases?**

In [0]:
gold_fraud_by_amount_bucket_df = (transactions_df    .withColumn(        "amount_bucket",        F.when(F.col("amount") < 25, "Low (< $25)")         .when((F.col("amount") >= 25) & (F.col("amount") < 100), "Medium ($25-$100)")         .when((F.col("amount") >= 100) & (F.col("amount") < 500), "High ($100-$500)")         .otherwise("Very High ($500+)")    )    .groupBy("amount_bucket")    .agg(        F.count("*").alias("total_txn_count"),        F.sum(F.col("is_fraud").cast("int")).alias("fraud_txn_count")    )    .withColumn("fraud_rate", F.col("fraud_txn_count") / F.col("total_txn_count"))    .orderBy(F.desc("fraud_rate")))gold_fraud_by_amount_bucket_df.write.mode("overwrite").saveAsTable("catalog.gold.fraud_by_amount_bucket")display(gold_fraud_by_amount_bucket_df)